# CoachPeaking TRIMP Gradient Boosting Pilot

This notebook runs a first time-aware experiment on the reviewed 2025 Garmin feature dataset. It is exploratory: the development period is used only for evaluation.

## Proposed feature set

The pilot keeps a small set to avoid obvious redundancy: `DURATION_SECONDS`, `AVERAGE_SPEED_MPS`, `AVERAGE_RUNNING_CADENCE_SPM`, and the shares for heart-rate zones 2 to 5. Each share is zone time divided by duration. Zone 1 is omitted because all five shares sum to one. Distance, maximum speed, average/max heart rate, Garmin training load, and Garmin training effects are excluded from this first model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATASET_PATH = Path('data/coachpeaking-trimp-dataset/2025/working/TRIMP_TRAIN_REVIEW.csv')
if not DATASET_PATH.exists():
    DATASET_PATH = Path('..') / DATASET_PATH
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Review dataset not found: {DATASET_PATH}')


In [ ]:
data = pd.read_csv(DATASET_PATH, parse_dates=['ACTIVITY_START_DATE'])
data = data.sort_values('ACTIVITY_START_DATE').reset_index(drop=True)
target_column = 'COACHPEAKING_TRIMP'
if data[target_column].isna().any():
    raise ValueError('The review dataset still contains missing TRIMP labels.')

for zone in range(1, 6):
    data[f'HR_ZONE_{zone}_SHARE'] = data[f'HR_TIME_IN_ZONE_{zone}'] / data['DURATION_SECONDS']

features = [
    'DURATION_SECONDS', 'AVERAGE_SPEED_MPS', 'AVERAGE_RUNNING_CADENCE_SPM',
    'HR_ZONE_2_SHARE', 'HR_ZONE_3_SHARE', 'HR_ZONE_4_SHARE', 'HR_ZONE_5_SHARE',
]
if data[features].isna().any().any():
    raise ValueError(f'Missing values in pilot features: {data[features].columns[data[features].isna().any()].tolist()}')
data[features + [target_column]].describe().T


## Sequential train/development split

The earliest 80% of activities is the training set and the latest 20% is the development set. No random shuffle is used.

In [ ]:
split_index = int(len(data) * 0.80)
train_data, dev_data = data.iloc[:split_index].copy(), data.iloc[split_index:].copy()
X_train, y_train = train_data[features], train_data[target_column]
X_dev, y_dev = dev_data[features], dev_data[target_column]
print(f'Train: {len(train_data)} rows, {train_data.ACTIVITY_START_DATE.min().date()} to {train_data.ACTIVITY_START_DATE.max().date()}')
print(f'Dev:   {len(dev_data)} rows, {dev_data.ACTIVITY_START_DATE.min().date()} to {dev_data.ACTIVITY_START_DATE.max().date()}')


## Gradient Boosting training and development evaluation

The hyperparameters are deliberately conservative for a small dataset. The resulting development metrics are a first signal, not a final estimate of production performance.

In [ ]:
model = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.05, max_depth=2,
    min_samples_leaf=5, loss='huber', random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
predictions = model.predict(X_dev)
metrics = pd.Series({
    'MAE': mean_absolute_error(y_dev, predictions),
    'RMSE': mean_squared_error(y_dev, predictions) ** 0.5,
    'MAPE_percent': ((y_dev - predictions).abs() / y_dev).mean() * 100,
    'R2': r2_score(y_dev, predictions),
}).round(3)
metrics


In [ ]:
evaluation = dev_data[['ACTIVITY_START_DATE', target_column]].copy()
evaluation['PREDICTED_TRIMP'] = predictions
evaluation['ABSOLUTE_ERROR'] = (evaluation[target_column] - evaluation['PREDICTED_TRIMP']).abs()
display(evaluation)

importance = pd.Series(model.feature_importances_, index=features).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_dev, predictions)
limits = [min(y_dev.min(), predictions.min()), max(y_dev.max(), predictions.max())]
axes[0].plot(limits, limits, 'k--', linewidth=1)
axes[0].set(xlabel='Observed TRIMP', ylabel='Predicted TRIMP', title='Development predictions')
importance.plot.barh(ax=axes[1], title='Feature importance')
axes[1].set_xlabel('Relative importance')
plt.tight_layout()
